In [1]:
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup

url = "https://www.ccny.cuny.edu/registrar/fall"
local_file = "ccny_fall_2021.html"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# Fetch page or load local copy if blocked
try:
    response = requests.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
        html_content = response.text
        print("Successfully fetched live HTML.")
    else:
        raise requests.RequestException(
            f"Blocked by host with status {response.status_code}"
        )
except Exception as e:
    print(f"Live request notice ({e}). Falling back to local saved copy...")
    with open(local_file, "r", encoding="utf-8") as f:
        html_content = f.read()
    print("Loaded HTML from local copy successfully.")

soup = BeautifulSoup(html_content, "html.parser")
print("BeautifulSoup object initialized.")

Live request notice (Blocked by host with status 403). Falling back to local saved copy...
Loaded HTML from local copy successfully.
BeautifulSoup object initialized.


In [2]:
table = soup.find("table")

raw_rows = []

if table is None:
    print("Error: Table not found on page.")
else:
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")

        # Keep rows that have 3 columns: Date, Day of Week, Description
        if len(tds) == 3:
            cols = [td.get_text(separator=" ", strip=True) for td in tds]

            # Exclude repeated table headers
            if cols[0].lower() not in ["date", "dates"]:
                raw_rows.append(cols)

    print("Total rows extracted:", len(raw_rows))
    print("First row:", raw_rows[0])
    print("Last row:", raw_rows[-1])

Total rows extracted: 36
First row: ['August 01', 'Sunday', 'Application for degree for January and February 2022 begins']
Last row: ['January 1, 2022', 'Saturday', 'College Closed; Fall 2021 Degree Conferral Date;']


In [3]:
import datetime

parsed_records = []

for date_str, dow, text in raw_rows:
    # Use the starting date if the entry represents a date range
    start_date_part = date_str.split("-")[0].strip()

    # Determine the correct year: January entries belong to 2022, everything else to 2021
    if "2022" in start_date_part:
        year = 2022
        clean_date = start_date_part.replace("2022", "").replace(",", "").strip()
    elif start_date_part.startswith("January"):
        year = 2022
        clean_date = start_date_part
    else:
        year = 2021
        clean_date = start_date_part.replace("2021", "").replace(",", "").strip()

    # Convert to python datetime.date object
    full_date_str = f"{clean_date} {year}"
    date_obj = pd.to_datetime(full_date_str).date()

    parsed_records.append({"date": date_obj, "dow": dow, "text": text})

# Construct DataFrame and set index
df = pd.DataFrame(parsed_records)
df.set_index("date", inplace=True)

df.head()

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply


In [5]:
# Verify column names and index type
print("DataFrame columns:", df.columns.tolist())
print("Index sample value:", df.index[0])
print("Type of index object:", type(df.index[0]))

DataFrame columns: ['dow', 'text']
Index sample value: 2021-08-01
Type of index object: <class 'datetime.date'>
